# Chapter 4 - Retrieval

The techniques from the chapter's *Reranking and hybrid search* and *Advanced
retrieval* sections, in the order the book introduces them, measured on the
Japan travel corpus.

Run `python build_corpus.py` once before this notebook.

Nothing here needs a paid API key. The embedding model is a static CPU model;
swap in `init_embeddings("openai:text-embedding-3-small")` if you prefer hosted.

## Setup

In [1]:
import json, os, re, time
import numpy as np

with open(os.path.join('data', 'japan_corpus.json'), encoding='utf-8') as f:
    corpus = json.load(f)

sections = corpus['sections']
places   = corpus['places']
print(f"{len(sections)} prose sections, {len(places)} places, "
      f"{sum(1 for p in places if p['lat'] is not None)} with coordinates")

550 prose sections, 1753 places, 1105 with coordinates


Two indexes, so we can measure what putting the structured places in costs and buys.
`prose_only` is what a naive markup-stripping parser leaves you with.

In [2]:
prose_only = [(s['path'] + '. ' + s['text'], s['doc_id']) for s in sections]
with_places = prose_only + [(p['text'], p['doc_id']) for p in places if p['name']]
print(f'prose only : {len(prose_only)} entries')
print(f'with places: {len(with_places)} entries')

prose only : 550 entries
with places: 2303 entries


### The embedding model

A static embedding model runs on a CPU in milliseconds and needs no key, which
keeps this notebook runnable anywhere. The hosted alternative is commented out.

In [3]:
from model2vec import StaticModel

_model = StaticModel.from_pretrained('minishlab/potion-retrieval-32M')

def embed(texts):
    return _model.encode(list(texts))

# Hosted alternative, one line, same interface downstream:
# from langchain.embeddings import init_embeddings
# _emb = init_embeddings('openai:text-embedding-3-small')
# def embed(texts): return np.array(_emb.embed_documents(list(texts)))

print('embedding dimension:', embed(['hello']).shape[1])

/private/tmp/claude-502/-Users-ben-Dropbox-projects-rag-book/40395c66-a409-41b0-831a-53797d33cbcb/scratchpad/v1env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 1239.45it/s]

embedding dimension: 512


## Reciprocal rank fusion

**Listing 4.1.** Merging two ranked lists on position alone, so an unbounded BM25
score and a cosine similarity never have to be put on a common scale.

In [4]:
def reciprocal_rank_fusion(*ranked_lists, k: int = 60, top_n: int = 10):
    """Merge ranked lists of document ids using their positions only."""
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)[:top_n]

# the keyword search and the vector search disagree about the best result
keyword_hits = ['visa_2019', 'visa_current', 'bali_guide']
vector_hits = ['visa_current', 'bali_guide', 'health_current']
print(reciprocal_rank_fusion(keyword_hits, vector_hits, top_n=3))

['visa_current', 'bali_guide', 'visa_2019']


Appearing respectably in both lists beats winning one of them outright, which is
what you want when one retriever is confidently wrong.

## A hybrid retriever

**Listing 4.2.** Keyword and vector search over the same texts, fused on rank.

In [5]:
import re

import numpy as np
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    return [t for t in re.split(r'\W+', text.lower()) if len(t) > 1]

class HybridRetriever:
    """Keyword and vector search over the same texts, fused on rank."""

    def __init__(self, texts: list[str], embed_fn):
        self.texts = texts
        self.embed = embed_fn
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])
        vectors = np.asarray(embed_fn(texts), dtype=float)
        self.vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

    def _keyword(self, query, n):
        scores = self.bm25.get_scores(tokenize(query))
        return [str(i) for i in np.argsort(-scores)[:n] if scores[i] > 0]

    def _vector(self, query, n):
        q = np.asarray(self.embed([query])[0], dtype=float)
        similarity = self.vectors @ (q / np.linalg.norm(q))
        return [str(i) for i in np.argsort(-similarity)[:n]]

    def search(self, query: str, k: int = 5, mode: str = 'hybrid') -> list[int]:
        if mode == 'vector':
            return [int(i) for i in self._vector(query, k)]
        if mode == 'bm25':
            return [int(i) for i in self._keyword(query, k)]
        merged = reciprocal_rank_fusion(self._keyword(query, k * 3),
                                        self._vector(query, k * 3), top_n=k)
        return [int(i) for i in merged]

Each arm over-fetches three times what we need before merging, so a document
ranked eighth by one retriever and second by the other can still surface.

## Measuring it

Sixteen questions that name a specific place, scored objectively: did a retrieved
entry literally contain the name? This is Figure 4.3 in the chapter.

In [6]:
ENTITY_QUERIES = [
    ('Where is Kenrokuen garden?', 'kenroku'),
    ('Where is the Toshogu shrine?', 'tosho'),
    ('Which district is Dotonbori in?', 'dotonbori'),
    ('Where is the golden pavilion Kinkaku-ji?', 'kinkaku'),
    ('Where is Tsukiji fish market?', 'tsukiji'),
    ('Where is the floating torii gate?', 'itsukushima'),
    ('Where is Nakamise shopping street?', 'nakamise'),
    ('Where is Gion geisha district?', 'gion'),
    ('Where is Odori park?', 'odori'),
    ('Where is Sanjusangendo temple?', 'sanjusangen'),
    ('Where is Ohori park?', 'ohori'),
    ('Where is Nijo castle?', 'nijo'),
    ('Where is Meiji shrine?', 'meiji'),
    ('Where is Todaiji temple?', 'todai'),
    ('Where is Himeji castle?', 'himeji'),
    ('Where is the Adachi museum?', 'adachi'),
]

def hit_at_k(retriever, cases, mode, k):
    hits = 0
    for query, term in cases:
        found = retriever.search(query, k=k, mode=mode)
        if any(term in retriever.texts[i].lower() for i in found):
            hits += 1
    return hits / len(cases)

In [7]:
import pandas as pd

rows = []
for label, index in [('prose only', prose_only), ('prose + places', with_places)]:
    retriever = HybridRetriever([t for t, _ in index], embed)
    for mode in ('vector', 'bm25', 'hybrid'):
        rows.append({
            'index': label, 'retriever': mode,
            'hit@1': hit_at_k(retriever, ENTITY_QUERIES, mode, 1),
            'hit@5': hit_at_k(retriever, ENTITY_QUERIES, mode, 5),
        })
pd.DataFrame(rows).pivot(index='retriever', columns='index').round(2)

hit@1                     hit@5           
index     prose + places prose only prose + places prose only
retriever                                                    
bm25                0.94       0.62           0.94       0.75
hybrid              0.88       0.50           0.88       0.75
vector              0.50       0.31           0.81       0.69

Two things to read off this. Every retriever improves when the index gains the
place records, and it improves more than any change of retriever does. Within
each index, vector search alone is weakest, because these questions name a place
and a name is a keyword.

## Retrieval presets

**Listing 4.4.** Bundles of settings chosen by what kind of question came in.
Routing without an agent.

In [8]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Preset:
    """Retrieval settings for one kind of question."""
    k: int
    chunk_size: int
    min_similarity: float
    temperature: float

PRESETS = {
    # a specific fact: few chunks, tight, no invention
    'lookup': Preset(k=5, chunk_size=400, min_similarity=0.30, temperature=0.0),
    # open-ended: more context, looser bar, a little freedom
    'synthesis': Preset(k=15, chunk_size=700, min_similarity=0.20, temperature=0.3),
    # browsing: cast wide, let the reranker sort it out
    'browse': Preset(k=20, chunk_size=500, min_similarity=0.10, temperature=0.0),
}

for name, preset in PRESETS.items():
    print(f'{name:10s} {preset}')

lookup     Preset(k=5, chunk_size=400, min_similarity=0.3, temperature=0.0)
synthesis  Preset(k=15, chunk_size=700, min_similarity=0.2, temperature=0.3)
browse     Preset(k=20, chunk_size=500, min_similarity=0.1, temperature=0.0)


## Relevance against variety

**Listing 4.4a.** A similarity search returns what matches best, which for
"three days in Kyoto" is five paragraphs about temples. Maximum marginal
relevance scores each candidate for relevance *and* for how much it repeats what
is already selected.

In [9]:
def mmr(query, texts, embed_fn, k=5, fetch_k=30, lambda_mult=0.5):
    """Greedy maximum marginal relevance over an already-embedded pool."""
    pool = texts[:fetch_k]
    V = np.asarray(embed_fn(pool), dtype=float)
    V /= np.linalg.norm(V, axis=1, keepdims=True)
    q = np.asarray(embed_fn([query])[0], dtype=float); q /= np.linalg.norm(q)
    relevance = V @ q
    chosen = [int(np.argmax(relevance))]
    while len(chosen) < min(k, len(pool)):
        redundancy = (V @ V[chosen].T).max(axis=1)
        score = lambda_mult * relevance - (1 - lambda_mult) * redundancy
        score[chosen] = -np.inf
        chosen.append(int(np.argmax(score)))
    return [pool[i] for i in chosen]

kyoto = [t for t, d in with_places if 'kyoto' in d][:60]
for lam, label in ((1.0, 'pure relevance'), (0.4, 'diversified')):
    picks = mmr('three days in Kyoto', kyoto, embed, k=4, lambda_mult=lam)
    print(f'lambda_mult={lam}  ({label})')
    for p in picks:
        print('   ', p[:76].replace(chr(10), ' '))

lambda_mult=1.0  (pure relevance)
    Kyoto > Do > Film industry. Kyoto is the traditional home of the Japanese fi
    Kyoto > Understand. Nestled among the mountains of the Kansai region of West
    Kyoto > See > World Heritage Sites. In 1994, 17 historic sites were inscribe
    Kyoto > Through Kansai International Airport > By bus. Comfortable limousine
lambda_mult=0.4  (diversified)
    Kyoto > Do > Film industry. Kyoto is the traditional home of the Japanese fi
    Kyoto > See. Kyoto offers an incredible number of attractions for tourists, 
    Kyoto > Get around > By subway. There are two subway lines which only serve 
    Kyoto > Understand > Climate. Kyoto truly exhibits the four seasons of sprin


At `lambda_mult=1.0` you get pure relevance. Lower it and each pick has to earn
its place against what is already chosen. `fetch_k` matters too: diversification
can only choose among candidates it was given.

### Two filters worth wiring in on day one

Serving a superseded document and leaking an internal one are the same bug: a
fact that belongs in the query got applied afterwards, or not at all.

In [10]:
# freshness: put the fact in metadata, filter inside the query
# retriever = vector_store.as_retriever(
#     search_kwargs={'k': 5, 'filter': {'is_current': True}})

# access control: same shape, and it has to be in the query, because
# filtering afterwards means the document was already read, logged and cached
# def retriever_for(user):
#     return vector_store.as_retriever(search_kwargs={
#         'k': 5,
#         'filter': {'tenant_id': user.tenant_id, 'audience': user.audience}})

# Demonstrated here without a store, on the Meridian corpus, which carries
# is_current and audience on every document.
from meridian import load_corpus
meridian = load_corpus()
def visible(docs, audience='customer'):
    return [d for d in docs
            if d.get('is_current', True) and d.get('audience') != 'agent'
            or audience == 'agent']
print(f'{len(meridian)} documents, {len(visible(meridian))} visible to a customer')
for d in meridian:
    if not d.get('is_current', True) or d.get('audience') == 'agent':
        print(f"   withheld: {d['doc_id']:26s} current={d.get('is_current')} "
              f"audience={d.get('audience')}")

22 documents, 18 visible to a customer
   withheld: baggage_2019_outdated      current=False audience=customer
   withheld: health_covid_archived      current=False audience=customer
   withheld: sop_complaints             current=True audience=agent
   withheld: sop_refunds                current=True audience=agent


## Indexing in levels

**Listing 4.5.** Splitting a document on its own headings. No clustering, no model.

The `title` carried into every section is the part that is easy to leave out and
expensive to leave out: without it, every destination's *Best time to visit*
section summarises identically and the first level cannot tell them apart.

In [11]:
HEADING = re.compile(r'^#{1,6} ', re.M)

def split_into_sections(doc: dict) -> list[dict]:
    """Split a document on its own headings, keeping the parent title."""
    text = doc['text']
    starts = [m.start() for m in HEADING.finditer(text)]
    if not starts:
        return [{'doc_id': doc['doc_id'], 'title': doc['title'],
                 'heading': '', 'text': text}]
    bounds = starts + [len(text)]
    out = []
    for i, start in enumerate(starts):
        block = text[start:bounds[i + 1]].strip()
        heading = block.splitlines()[0].lstrip('# ').strip()
        out.append({'doc_id': doc['doc_id'], 'title': doc['title'],
                    'heading': heading, 'text': block})
    return out

demo = {'doc_id': 'kyoto', 'title': 'Kyoto, Japan',
        'text': '## Overview\nKyoto was the capital for a thousand years.\n'
                '## Best time to visit\nPeak season is April and November.'}
for s in split_into_sections(demo):
    print(f"{s['title']} > {s['heading']}: {s['text'][:44]}...")

Kyoto, Japan > Overview: ## Overview
Kyoto was the capital for a thou...
Kyoto, Japan > Best time to visit: ## Best time to visit
Peak season is April a...


### Does the extra level pay?

The chapter says no, on this corpus, and gives two reasons. Both are measurable.

In [12]:
# Does a second level have anything to narrow? Two properties decide it.
from collections import Counter

per_doc = Counter(s['doc_id'] for s in sections)
sections_per_doc = sum(per_doc.values()) / len(per_doc)

headings = [s['path'].split(' > ')[-1].lower() for s in sections]
counts = Counter(headings)
shared = sum(n for h, n in counts.items() if n > 1)

print(f'sections per document      : {sections_per_doc:.1f}')
mean_len = np.mean([len(s['text']) for s in sections])
print(f'mean section length (chars): {mean_len:.0f}')
print(f'headings shared across docs: {shared}/{len(headings)} = {shared/len(headings):.0%}')
print(f'most repeated              : {counts.most_common(4)}')

sections per document      : 12.0
mean section length (chars): 816
headings shared across docs: 442/550 = 80%
most repeated              : [('by train', 36), ('understand', 32), ('by bus', 31), ('eat', 31)]


In [13]:
# Build cost and query cost are different things. Separate them.
flat = HybridRetriever([t for t, _ in with_places], embed)   # one-time

t0 = time.time()
for query, _ in ENTITY_QUERIES:
    flat.search(query, k=5, mode='vector')
per_query = (time.time() - t0) / len(ENTITY_QUERIES) * 1000
print(f'flat vector search over {len(with_places)} entries: {per_query:.1f}ms per query')

flat vector search over 2303 entries: 1.3ms per query


A second level can only narrow if sections hold several entries, and can only
pick the right section if headings distinguish one document from another.
Neither holds here, and flat search is already fast. Levels start paying at
roughly ten thousand entries, or when scope isolation between tenants has to be
provable rather than merely likely.

## Checking your own work

**Listing 4.9.** Verifying citations is string matching, not a model problem.
It costs microseconds and catches invented sources outright.

In [14]:
def verify_citations(answer: str, retrieved: dict[str, str]) -> list[str]:
    """Return the problems with an answer's citations, empty if it is clean."""
    cited = set(re.findall(r'\[source: ([^\]]+)\]', answer))
    problems = []
    for source_id in cited:
        if source_id not in retrieved:
            problems.append(f'cites {source_id}, which was never retrieved')
    if not cited:
        problems.append('no citations at all')
    return problems

retrieved = {'jp_kyoto': 'Kyoto was the capital for a thousand years.'}
print(verify_citations('Kyoto was the capital [source: jp_kyoto].', retrieved))
print(verify_citations('Kyoto was the capital [source: jp_osaka].', retrieved))
print(verify_citations('Kyoto was the capital.', retrieved))

[]
['cites jp_osaka, which was never retrieved']
['no citations at all']


**Listing 4.10.** The groundedness gate. This needs a chat model, so it is left
as a function rather than run here.

It survives the finding that models cannot usefully self-correct, because it is
not self-reflection: the model compares an answer against text it did not write.

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GROUNDED = ChatPromptTemplate.from_template(
    'Is every claim in the answer supported by the context? '
    'Reply with one word, YES or NO.\n\n'
    'Context:\n{context}\n\nAnswer:\n{answer}'
)

def is_grounded(answer, chunks, model) -> bool:
    context = '\n\n'.join(c.page_content for c in chunks)
    verdict = (GROUNDED | model | StrOutputParser()).invoke(
        {'context': context[:4000], 'answer': answer}
    )
    return verdict.strip().upper().startswith('Y')

# from langchain.chat_models import init_chat_model
# model = init_chat_model('openai:gpt-4o-mini')
# is_grounded('Kyoto was the capital.', retrieved_docs, model)
print('defined; needs a chat model to run')

defined; needs a chat model to run


## Where this goes next

Ranking gains run out. `04_itinerary_assistant.ipynb` picks up the other half of
the argument: what else belongs in the pipeline besides prose and a ranker.